# Topic 1: The Retriever (done right)
---

Foundation: what you already have
In Step 4 you built a vectorstore (FAISS/Chroma) that stores each chunk with its vector and answers similarity_search(query, k). The retriever is a super thin layer on that — it exposes one standard method, invoke(query), that returns the top-k chunk Documents.


The whole point of the abstraction: any store can become a retriever, and any retriever plugs into the same downstream code.

This is the key insight for Topic 1:

```retriever = store.as_retriever(...)```

- store is a FAISS object
- store.as_retriever() returns a VectorStoreRetriever
- You call retriever.invoke(query) and get back a list of Documents
That's literally the entire topic at its core. invoke() is similarity_search under the hood — same vectors, same math — just presented through one uniform method.

The two knobs you control
*Knob A: how many chunks (k).*

```store.as_retriever(search_kwargs={"k": 5})   # return 5 chunks```

Default is k=4.


*Knob B: which search type.* This is subtler — vector stores can retrieve in different ways:
1. similarity (default) — cosine/L2 of the query vs each chunk, take top-k
2. similarity_score_threshold — like above, but drop chunks below a score threshold (avoids junk)
Let me show you the threshold version, because it fixes a real weakness we already saw (the duplicate-result bug):

```$ cd /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain && timeout 150 /home/vanitas/Python/git/elusive_Agentic_AI/venv/bin/python - <<'EOF' 2>&1 | grep -v DeprecationWarning | grep -v "fitz"
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

DATA = Path("data_ingestion")
emb = OllamaEmbeddings(model="nomic-embed-text")
split = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = split.split_documents(TextLoader(str(DATA/"heylog_eve_album.txt")).load())
store = FAISS.from_documents(chunks, emb)

query = "shame like Adam and Eve"

# --- default similarity: top-k, no quality gate ---
plain = store.as_retriever(search_kwargs={"k": 4})
print("=== similarity (k=4) ===")
hits = plain.invoke(query)
print("got", len(hits), "chunks")
for d in hits:
    print("  |", d.page_content[:48].replace("\n", " "))
EOF
=== similarity (k=4) ===
got 4 chunks
  | I try to hide, but it's not gonna help or save m
  | I try to hide, but it's not gonna help or save m
  | "cover up all the shame like Adam and Eve" ties 
  | [Outro] Oh, oh-oh, oh Oh, oh-oh, oh (Like Adam a
```

There's the duplicate bug you noticed — the exact same "I try to hide..." line comes back twice because that lyric repeats verbatim in the album, so two chunks embed to nearly-identical vectors and both rank in the top 4. This wastes LLM context and could bias the answer.
Now the threshold variant:

```
$ cd /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain && timeout 150 /home/vanitas/Python/git/elusive_Agentic_AI/venv/bin/python - <<'EOF' 2>&1 | grep -v DeprecationWarning | grep -v "fitz"
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

DATA = Path("data_ingestion")
emb = OllamaEmbeddings(model="nomic-embed-text")
split = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = split.split_documents(TextLoader(str(DATA/"heylog_eve_album.txt")).load())
store = FAISS.from_documents(chunks, emb)

# -------------------------------
# NOTE on FAISS scores:
# FAISS similarity_search_with_score returns L2 DISTANCE (lower = better),
# ~0 = identical, large = unrelated. All values are typically > 0.
# -------------------------------

# See the actual scores so we can pick a sensible threshold
scored = store.similarity_search_with_score("shame like Adam and Eve", k=6)
print("=== raw scores (L2 distance, lower=more similar) ===")
for d, s in scored:
    print(f"  score {s:.3f} | {d.page_content[:40].replace(chr(10),' ')}")
EOF
=== raw scores (L2 distance, lower=more similar) ===
  score 0.494 | I try to hide, but it's not gonna help o
  score 0.494 | I try to hide, but it's not gonna help o
  score 0.543 | "cover up all the shame like Adam and Ev
  score 0.636 | [Outro] Oh, oh-oh, oh Oh, oh-oh, oh (Lik
  score 0.702 | Oh, oh-oh, oh (Eve, Eve)
  score 0.758 | The album is approached biblically, focu

```

Now I can see the scores. Top-2 identical at 0.494 (the duplicates), then 0.543, 0.636, 0.702, 0.758.


The threshold version uses score_threshold — with FAISS where lower distance is better, the threshold means "only keep chunks with distance ≤ this." But there's a subtlety: similarity_score_threshold in LangChain's FAISS wrapper actually inverts the distance into a 0-1 similarity (higher = better), and filters on that. Let me demonstrate the correct usage and verify whether it removes the near-duplicate:

```
$ cd /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain && timeout 150 /home/vanitas/Python/git/elusive_Agentic_AI/venv/bin/python - <<'EOF' 2>&1 | grep -v DeprecationWarning | grep -v "fitz"
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
import inspect
from langchain_community import vectorstores

DATA = Path("data_ingestion")
emb = OllamaEmbeddings(model="nomic-embed-text")
split = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = split.split_documents(TextLoader(str(DATA/"heylog_eve_album.txt")).load())
store = FAISS.from_documents(chunks, emb)

# The FAISS wrapper's threshold search uses _normalize_distance -> similarity
# higher = more similar, threshold is a lower-bound on similarity (0..1)
def inv_dist_to_sim(d):
    # FAISS: similarity = 1/(1+d) roughly, used by langchain
    return 1.0 / (1.0 + d)

for s in [0.494, 0.543, 0.636, 0.702, 0.758]:
    print(f"  dist {s:.3f} -> sim ~{inv_dist_to_sim(s):.3f}")

# use similarity_score_threshold with a similarity score (0..1)
retriever = store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.65, "k": 4},
)
print("\n=== similarity_score_threshold (sim >= 0.65) ===")
hits = retriever.invoke("shame like Adam and Eve")
print("got", len(hits), "chunks")
for d in hits:
    print("  |", d.page_content[:48].replace("\n", " "))
EOF
dist 0.494 -> sim ~0.669
  dist 0.543 -> sim ~0.648
  dist 0.636 -> sim ~0.611
  dist 0.702 -> sim ~0.588
  dist 0.758 -> sim ~0.569

=== similarity_score_threshold (sim >= 0.65) ===
got 2 chunks
  | I try to hide, but it's not gonna help or save m
  | I try to hide, but it's not gonna help or save m
```

This reveals something important. The threshold (sim ≥ 0.65) kept only the 2 duplicates (0.669 similarity) and dropped the "cover up all the shame" chunk (0.648).


So the threshold didn't fix the duplicate here — because the duplicate is the most similar match. The score_threshold is a quality floor (drop weak matches), not a dedup tool. It correctly dropped the weaker stuff (0.648 and below), but both near-identical copies scored above the floor, so both remained.


This is the teaching moment for Topic 1:
Three takeaways for an honest mental model of retrievers


1. **invoke(query) = similarity_search with a standard signature.** The retriever is just an interface; the math is identical to what you already did in the vectorstore step.
2. **k controls quantity, score_threshold controls quality floor.**They're independent. k says "return up to this many." score_threshold says "but only if good enough." Neither deduplicates.
3. **Duplicates are a real problem the retriever alone does NOT solve.**Two near-identical chunks both survive because both are good matches. Fixing that requires deduplication in the RAG chain — which is part of Topic 5 (provenance/debugging), not the retriever itself.

Interesting — even MMR (which is specifically designed to reduce redundancy) still returned the duplicate here, because lambda_mult=0.85 heavily weights similarity over diversity. If I lower lambda_mult, diversity wins and the duplicate gets dropped. This confirms the deeper point: retrievers have knobs, but none cleanly solve dedup by default — you have to know which knob to turn, and sometimes you still need post-processing in the chain.
That's the honest end of Topic 1.

Topic 1 complete — the summary
The Retriever, in one mental model:
- store.as_retriever(...) returns a VectorStoreRetriever — a uniform invoke(query) → list[Document] interface over your vectorstore.
- invoke() = similarity_search under the hood. Same vectors, same math.
- Knobs:
- search_kwargs={"k": n} → how many chunks
- search_type="similarity" (default) | "similarity_score_threshold" (quality floor) | "mmr" (diversity)
- search_type="mmr" takes fetch_k (how many to consider) + lambda_mult (similarity vs diversity weight)
- Critical reality: none of these deduplicate out-of-the-box. Duplicate near-identical chunks survived even MMR here. That's not a retriever bug — it's the reason Topic 5 (provenance/debugging) exists, where we'll dedup in the chain.
On your data: we saw the same paranoid lyric line come back 2× because it repeats verbatim in the album and embeds to a near-identical vector.

In [ ]:
# ---- Topic 1: build the vectorstore ONCE; later cells reuse it ----
# The retriever wraps the store, so we need the same FAISS store from
# Step 4. Data lives in the sibling folder section_5_LangChain.
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# robust path finding: works no matter which folder Jupyter opened the notebook in
def find_album():
    roots = [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for cand in (
            root / "data_ingestion" / "heylog_eve_album.txt",
            root / "section_5_LangChain" / "data_ingestion" / "heylog_eve_album.txt",
        ):
            if cand.exists():
                return cand
    raise FileNotFoundError("heylog_eve_album.txt not found")

emb = OllamaEmbeddings(model="nomic-embed-text")
split = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = split.split_documents(TextLoader(str(find_album())).load())
store = FAISS.from_documents(chunks, emb)
print("store  :", type(store).__name__, "| chunks:", store.index.ntotal)

# The retriever IS a VectorStoreRetriever object:
# as_retriever() is just a method ON THE STORE that returns one.
retriever = store.as_retriever(search_kwargs={"k": 3})
print("retriever:", type(retriever).__name__)
print("instance of VectorStoreRetriever:", isinstance(retriever, VectorStoreRetriever))

# One uniform method is all you call:
hits = retriever.invoke("thrown away and forgotten by someone")
print("invoke() returned", len(hits), "Documents")
for d in hits:
    print("  |", d.page_content[:55].replace("\n", " "))

In [ ]:
# ---- Knob A: k controls HOW MANY chunks are returned ----
# Default is k=4; plain similarity just takes the top-k by distance.
query = "shame like Adam and Eve"

plain = store.as_retriever(search_kwargs={"k": 4})
hits = plain.invoke(query)
print("=== similarity (k=4) ===")
print("got", len(hits), "chunks")
for d in hits:
    print("  |", d.page_content[:48].replace("\n", " "))

# NOTE: the SAME lyric line comes back twice -- the "I try to hide" verse
# repeats verbatim in the album, so two chunks embed to near-identical
# vectors and both rank in the top-k. Classic retrieval duplication.

In [ ]:
# ---- Look at the underlying scores ----
# similarity_search_with_score returns (document, L2 distance) pairs.
# LOWER distance = MORE similar; ~0 = identical; large = unrelated.
scored = store.similarity_search_with_score("shame like Adam and Eve", k=6)
print("=== raw scores (L2 distance, lower=more similar) ===")
for d, s in scored:
    print(f"  score {s:.3f} | {d.page_content[:40].replace(chr(10), ' ')}")

# notice the two IDENTICAL top scores (0.494): those are the duplicates.

In [ ]:
# ---- Knob B: score_threshold drops WEAK matches (quality floor) ----
# FAISS returns L2 distance (lower=better). LangChain's threshold search
# inverts distance -> similarity (0..1, higher=better) with 1/(1+d).
def dist_to_sim(d):
    return 1.0 / (1.0 + d)

for s in [0.494, 0.543, 0.636, 0.702, 0.758]:
    print(f"  dist {s:.3f} -> sim ~{dist_to_sim(s):.3f}")

retriever = store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.65, "k": 4},
)
print("\n=== similarity_score_threshold (sim >= 0.65) ===")
hits = retriever.invoke("shame like Adam and Eve")
print("got", len(hits), "chunks")
for d in hits:
    print("  |", d.page_content[:48].replace("\n", " "))

# takeaway: the threshold is a QUALITY FLOOR, not a dedup tool -- both
# near-identical duplicates scored above the floor, so BOTH survived.

In [ ]:
# ---- Knob B: mmr adds DIVERSITY ----
# mmr = Maximal Marginal Relevance: pick chunks similar to the query but
# penalize ones too similar to already-chosen chunks.
#   fetch_k     : how many candidates to consider first
#   lambda_mult : 1.0 = pure similarity, 0.0 = pure diversity
retriever = store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.85},
)
print("=== search_type='mmr' (Maximal Marginal Relevance) ===")
hits = retriever.invoke("shame and hiding like Adam and Eve")
for d in hits:
    print("  |", d.page_content[:48].replace("\n", " "))

# even at 0.85 (heavily similarity-weighted) the duplicate crept in --
# that is exactly why Topic 5 (provenance / debugging) exists.